Our main focus is to train our model on the basis :-
1) What was the Total Score of first innigs 
2) How much the batting team has scored in how much balls(in 2nd innigs)
3) Seeing the situation the chasing team is in ,we predict there winnings and loosing chanches
4) We want to return probabilities as the result of winng or losig of the batting team that's why we are using logistic regression

In [427]:
import sklearn
print(sklearn.__version__)

1.6.1


In [428]:
import numpy as np
import pandas as pd

In [429]:
match = pd.read_csv('matches_updated_ipl_upto_2025.csv')
delivery = pd.read_csv('deliveries_updated_ipl_upto_2025.csv')

In [430]:
delivery['total_runs'] = delivery['batsman_runs']+delivery['extras']
delivery.head()

,matchId,inning,over_ball,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,...,extras,isWide,isNoBall,Byes,LegByes,Penalty,dismissal_kind,player_dismissed,date,total_runs
0,335982,1,0.1,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,...,1,NaN,NaN,NaN,1.0,NaN,NaN,NaN,2008-04-18,1
1,335982,1,0.2,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,0
2,335982,1,0.3,0,3,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,...,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,1
3,335982,1,0.4,0,4,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,0
4,335982,1,0.5,0,5,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,0


Data preparation 


We are going to train our model on the basis of deliveries.csv as it has ball by ball information of matches

But we are missing some columns:-
1) Winner of the match (label/output)
2) Total score of the first innigs and current score

In [431]:
delivery.head()

,matchId,inning,over_ball,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,...,extras,isWide,isNoBall,Byes,LegByes,Penalty,dismissal_kind,player_dismissed,date,total_runs
0,335982,1,0.1,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,...,1,NaN,NaN,NaN,1.0,NaN,NaN,NaN,2008-04-18,1
1,335982,1,0.2,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,0
2,335982,1,0.3,0,3,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,...,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,1
3,335982,1,0.4,0,4,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,0
4,335982,1,0.5,0,5,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,0


In [432]:
delivery.shape

(278205, 21)

In [433]:
match.head()

,season,venue,event,winner_runs,umpire2,toss_winner,date,neutralvenue,umpire1,city,...,team2,balls_per_over,winner_wickets,tv_umpire,player_of_match,match_referee,outcome,date2,match_number,matchId
0,2007/08,M Chinnaswamy Stadium,Indian Premier League,140.0,RE Koertzen,Royal Challengers Bangalore,2008-04-18,NaN,Asad Rauf,Bangalore,...,Kolkata Knight Riders,6,NaN,AM Saheba,BB McCullum,J Srinath,NaN,NaN,1.0,335982
1,2007/08,"Punjab Cricket Association Stadium, Mohali",Indian Premier League,33.0,SL Shastri,Chennai Super Kings,2008-04-19,NaN,MR Benson,Chandigarh,...,Chennai Super Kings,6,NaN,RB Tiffin,MEK Hussey,S Venkataraghavan,NaN,NaN,2.0,335983
2,2007/08,Feroz Shah Kotla,Indian Premier League,NaN,GA Pratapkumar,Rajasthan Royals,2008-04-19,NaN,Aleem Dar,Delhi,...,Rajasthan Royals,6,9.0,IL Howell,MF Maharoof,GR Viswanath,NaN,NaN,3.0,335984
3,2007/08,Eden Gardens,Indian Premier League,NaN,K Hariharan,Deccan Chargers,2008-04-20,NaN,BF Bowden,Kolkata,...,Deccan Chargers,6,5.0,Asad Rauf,DJ Hussey,FM Engineer,NaN,NaN,4.0,335986
4,2007/08,Wankhede Stadium,Indian Premier League,NaN,DJ Harper,Mumbai Indians,2008-04-20,NaN,SJ Davis,Mumbai,...,Royal Challengers Bangalore,6,5.0,AV Jayaprakash,MV Boucher,J Srinath,NaN,NaN,5.0,335985


Calculation of Total Score of first innigs

In [434]:
total_score_df = delivery.groupby(['matchId','inning']).sum()['total_runs'].reset_index()

In [435]:
total_score_df = total_score_df[total_score_df['inning'] == 1]

In [436]:
total_score_df

,matchId,inning,total_runs
0,335982,1,222
2,335983,1,240
4,335984,1,129
6,335985,1,165
8,335986,1,110
...,...,...,...
2355,1473508,1,101
2357,1473509,1,228
2359,1473510,1,203
2361,1473511,1,190


merging the total_score_df to match as we need to do some changes in our data like:-
1) Removing the teams like RPS ,GL 
2) Renaming teams who have changed their names 
3) Removing the matches having reduced overs due to rain

we are merging the total_score with match and not with delivery as the match dataset is way shorter an data proccessing will take less time 

In [437]:
match_df = match.merge(total_score_df[['matchId','total_runs']],left_on='matchId',right_on='matchId')

In [438]:
match_df

,season,venue,event,winner_runs,umpire2,toss_winner,date,neutralvenue,umpire1,city,...,balls_per_over,winner_wickets,tv_umpire,player_of_match,match_referee,outcome,date2,match_number,matchId,total_runs
0,2007/08,M Chinnaswamy Stadium,Indian Premier League,140.0,RE Koertzen,Royal Challengers Bangalore,2008-04-18,NaN,Asad Rauf,Bangalore,...,6,NaN,AM Saheba,BB McCullum,J Srinath,NaN,NaN,1.0,335982,222
1,2007/08,"Punjab Cricket Association Stadium, Mohali",Indian Premier League,33.0,SL Shastri,Chennai Super Kings,2008-04-19,NaN,MR Benson,Chandigarh,...,6,NaN,RB Tiffin,MEK Hussey,S Venkataraghavan,NaN,NaN,2.0,335983,240
2,2007/08,Feroz Shah Kotla,Indian Premier League,NaN,GA Pratapkumar,Rajasthan Royals,2008-04-19,NaN,Aleem Dar,Delhi,...,6,9.0,IL Howell,MF Maharoof,GR Viswanath,NaN,NaN,3.0,335984,129
3,2007/08,Eden Gardens,Indian Premier League,NaN,K Hariharan,Deccan Chargers,2008-04-20,NaN,BF Bowden,Kolkata,...,6,5.0,Asad Rauf,DJ Hussey,FM Engineer,NaN,NaN,4.0,335986,110
4,2007/08,Wankhede Stadium,Indian Premier League,NaN,DJ Harper,Mumbai Indians,2008-04-20,NaN,SJ Davis,Mumbai,...,6,5.0,AV Jayaprakash,MV Boucher,J Srinath,NaN,NaN,5.0,335985,165
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1164,2025,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Indian Premier League,NaN,VK Sharma,Royal Challengers Bengaluru,2025-05-27,NaN,MA Gough,Lucknow,...,6,6.0,UV Gandhe,JM Sharma,P Dharmani,NaN,NaN,70.0,1473507,227
1165,2025,Maharaja Yadavindra Singh International Cricke...,Indian Premier League,NaN,VK Sharma,Royal Challengers Bengaluru,2025-05-29,NaN,MA Gough,New Chandigarh,...,6,8.0,R Pandit,Suyash Sharma,V Narayan Kutty,NaN,NaN,NaN,1473508,101
1166,2025,Maharaja Yadavindra Singh International Cricke...,Indian Premier League,20.0,UV Gandhe,Mumbai Indians,2025-05-30,NaN,R Pandit,New Chandigarh,...,6,NaN,VK Sharma,RG Sharma,Prakash Bhatt,NaN,NaN,NaN,1473509,228
1167,2025,"Narendra Modi Stadium, Ahmedabad",Indian Premier League,NaN,Nitin Menon,Punjab Kings,2025-06-01,NaN,CB Gaffaney,Ahmedabad,...,6,5.0,KN Ananthapadmanabhan,SS Iyer,J Srinath,NaN,NaN,NaN,1473510,203


In [439]:
match_df['team1'].unique()

array(['Royal Challengers Bangalore', 'Kings XI Punjab',
       'Delhi Daredevils', 'Kolkata Knight Riders', 'Mumbai Indians',
       'Rajasthan Royals', 'Deccan Chargers', 'Chennai Super Kings',
       'Kochi Tuskers Kerala', 'Pune Warriors', 'Sunrisers Hyderabad',
       'Gujarat Lions', 'Rising Pune Supergiants',
       'Rising Pune Supergiant', 'Delhi Capitals', 'Punjab Kings',
       'Lucknow Super Giants', 'Gujarat Titans',
       'Royal Challengers Bengaluru'], dtype=object)

In [440]:
teams = [
    'Sunrisers Hyderabad',
    'Mumbai Indians',
    'Royal Challengers Bangalore',
    'Kolkata Knight Riders',
    'Punjab Kings',
    'Chennai Super Kings',
    'Rajasthan Royals',
    'Delhi Capitals',
    'Lucknow Super Giants',
    'Gujarat Titans'
]



In [441]:
match_df['team1'] = match_df['team1'].str.replace('Delhi Daredevils','Delhi Capitals')
match_df['team2'] = match_df['team2'].str.replace('Delhi Daredevils','Delhi Capitals')

match_df['team1'] = match_df['team1'].str.replace('Deccan Chargers','Sunrisers Hyderabad')
match_df['team2'] = match_df['team2'].str.replace('Deccan Chargers','Sunrisers Hyderabad')

match_df['team1'] = match_df['team1'].str.replace('Kings XI Punjab','Punjab Kings')
match_df['team2'] = match_df['team2'].str.replace('Kings XI Punjab','Punjab Kings')

match_df['team1'] = match_df['team1'].str.replace('Royal Challengers Bengaluru','Royal Challengers Bangalore')
match_df['team2'] = match_df['team2'].str.replace('Royal Challengers Bengaluru','Royal Challengers Bangalore')


In [442]:
match_df = match_df[match_df['team1'].isin(teams)]
match_df = match_df[match_df['team2'].isin(teams)]

In [443]:
match_df['venue'].unique()

array(['M Chinnaswamy Stadium',
       'Punjab Cricket Association Stadium, Mohali', 'Feroz Shah Kotla',
       'Eden Gardens', 'Wankhede Stadium', 'Sawai Mansingh Stadium',
       'Rajiv Gandhi International Stadium, Uppal',
       'MA Chidambaram Stadium, Chepauk', 'Dr DY Patil Sports Academy',
       'Newlands', "St George's Park", 'Kingsmead', 'SuperSport Park',
       'Buffalo Park', 'New Wanderers Stadium', 'De Beers Diamond Oval',
       'OUTsurance Oval', 'Brabourne Stadium',
       'Sardar Patel Stadium, Motera', 'Barabati Stadium',
       'Brabourne Stadium, Mumbai',
       'Vidarbha Cricket Association Stadium, Jamtha',
       'Himachal Pradesh Cricket Association Stadium',
       'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium',
       'Subrata Roy Sahara Stadium',
       'Shaheed Veer Narayan Singh International Stadium',
       'JSCA International Stadium Complex', 'Sheikh Zayed Stadium',
       'Sharjah Cricket Stadium', 'Dubai International Cricket Stadium',
      

In [444]:
# 1. Bengaluru
match_df['venue'] = match_df['venue'].str.replace('M.Chinnaswamy Stadium', 'M Chinnaswamy Stadium', regex=False)
match_df['venue'] = match_df['venue'].str.replace('M Chinnaswamy Stadium, Bengaluru', 'M Chinnaswamy Stadium', regex=False)

# 2. Mohali (Standardizing all to the updated IS Bindra name)
match_df['venue'] = match_df['venue'].str.replace('Punjab Cricket Association Stadium, Mohali', 'Punjab Cricket Association IS Bindra Stadium, Mohali', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh', 'Punjab Cricket Association IS Bindra Stadium, Mohali', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Punjab Cricket Association IS Bindra Stadium', 'Punjab Cricket Association IS Bindra Stadium, Mohali', regex=False)
# Fix the double "Mohali, Mohali" that might happen from the line above
match_df['venue'] = match_df['venue'].str.replace('Punjab Cricket Association IS Bindra Stadium, Mohali, Mohali', 'Punjab Cricket Association IS Bindra Stadium, Mohali', regex=False)

# 3. Delhi
match_df['venue'] = match_df['venue'].str.replace('Feroz Shah Kotla', 'Arun Jaitley Stadium', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Arun Jaitley Stadium, Delhi', 'Arun Jaitley Stadium', regex=False)

# 4. Chennai
match_df['venue'] = match_df['venue'].str.replace('MA Chidambaram Stadium, Chepauk, Chennai', 'MA Chidambaram Stadium', regex=False)
match_df['venue'] = match_df['venue'].str.replace('MA Chidambaram Stadium, Chepauk', 'MA Chidambaram Stadium', regex=False)

# 5. Hyderabad
match_df['venue'] = match_df['venue'].str.replace('Rajiv Gandhi International Stadium, Uppal, Hyderabad', 'Rajiv Gandhi International Stadium', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Rajiv Gandhi International Stadium, Uppal', 'Rajiv Gandhi International Stadium', regex=False)

# 6. Ahmedabad
match_df['venue'] = match_df['venue'].str.replace('Sardar Patel Stadium, Motera', 'Narendra Modi Stadium', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Narendra Modi Stadium, Ahmedabad', 'Narendra Modi Stadium', regex=False)

# 7. Pune
match_df['venue'] = match_df['venue'].str.replace('Subrata Roy Sahara Stadium', 'Maharashtra Cricket Association Stadium', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Maharashtra Cricket Association Stadium, Pune', 'Maharashtra Cricket Association Stadium', regex=False)

# 8. Mumbai (Wankhede, Brabourne, DY Patil)
match_df['venue'] = match_df['venue'].str.replace('Wankhede Stadium, Mumbai', 'Wankhede Stadium', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Brabourne Stadium, Mumbai', 'Brabourne Stadium', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Dr DY Patil Sports Academy, Mumbai', 'Dr DY Patil Sports Academy', regex=False)

# 9. Kolkata & Jaipur
match_df['venue'] = match_df['venue'].str.replace('Eden Gardens, Kolkata', 'Eden Gardens', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Sawai Mansingh Stadium, Jaipur', 'Sawai Mansingh Stadium', regex=False)

# 10. Dharamsala & Visakhapatnam
match_df['venue'] = match_df['venue'].str.replace('Himachal Pradesh Cricket Association Stadium, Dharamsala', 'Himachal Pradesh Cricket Association Stadium', regex=False)
match_df['venue'] = match_df['venue'].str.replace('Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam', 'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium', regex=False)

# 11. Abu Dhabi
match_df['venue'] = match_df['venue'].str.replace('Zayed Cricket Stadium, Abu Dhabi', 'Sheikh Zayed Stadium', regex=False)

# 12. Mullanpur
match_df['venue'] = match_df['venue'].str.replace('Maharaja Yadavindra Singh International Cricket Stadium, New Chandigarh', 'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur', regex=False)

In [445]:
match_df.shape

(1054, 29)

In [446]:
match_df = match_df[match_df['method'] != 'D/L']

After this data proccessing we can remove other columns as there work is done here and only keep the ones that are useful 

In [447]:
match_df = match_df[['matchId','venue','winner','total_runs']]

finally adding new columns like 'match_id','city','winner','total_runs' to delivery_df 

In [448]:
delivery_df = match_df.merge(delivery,on='matchId')

now considering only the 2nd innigs data 

In [449]:
delivery_df = delivery_df[delivery_df['inning'] == 2]

In [450]:
delivery_df

,matchId,venue,winner,total_runs_x,inning,over_ball,over,ball,batting_team,bowling_team,...,extras,isWide,isNoBall,Byes,LegByes,Penalty,dismissal_kind,player_dismissed,date,total_runs_y
124,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.1,0,1,Royal Challengers Bangalore,Kolkata Knight Riders,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,1
125,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.2,0,2,Royal Challengers Bangalore,Kolkata Knight Riders,...,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,1
126,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.3,0,3,Royal Challengers Bangalore,Kolkata Knight Riders,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,0
127,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.4,0,4,Royal Challengers Bangalore,Kolkata Knight Riders,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,1
128,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.5,0,5,Royal Challengers Bangalore,Kolkata Knight Riders,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-04-18,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248533,1473511,Narendra Modi Stadium,Royal Challengers Bengaluru,190,2,19.2,19,2,Punjab Kings,Royal Challengers Bengaluru,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-03,0
248534,1473511,Narendra Modi Stadium,Royal Challengers Bengaluru,190,2,19.3,19,3,Punjab Kings,Royal Challengers Bengaluru,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-03,6
248535,1473511,Narendra Modi Stadium,Royal Challengers Bengaluru,190,2,19.4,19,4,Punjab Kings,Royal Challengers Bengaluru,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-03,4
248536,1473511,Narendra Modi Stadium,Royal Challengers Bengaluru,190,2,19.5,19,5,Punjab Kings,Royal Challengers Bengaluru,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-03,6


Calculating the current score of 2nd innigs using the runs scored on each ball using cumulative sum

In [451]:
delivery_df['current_score'] = delivery_df.groupby('matchId')['total_runs_y'].cumsum()

Making new Columns like runs_left ,balls_left as models are bad at arithmetic it will give your model a great relation with the label/output

In [452]:
delivery_df['runs_left'] = delivery_df['total_runs_x'] - delivery_df['current_score']

In [453]:
delivery_df['balls_left'] = 120 - (delivery_df['over']*6 + delivery_df['ball'])

In [454]:
delivery_df

,matchId,venue,winner,total_runs_x,inning,over_ball,over,ball,batting_team,bowling_team,...,Byes,LegByes,Penalty,dismissal_kind,player_dismissed,date,total_runs_y,current_score,runs_left,balls_left
124,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.1,0,1,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,NaN,NaN,2008-04-18,1,1,221,119
125,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.2,0,2,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,NaN,NaN,2008-04-18,1,2,220,118
126,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.3,0,3,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,NaN,NaN,2008-04-18,0,2,220,117
127,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.4,0,4,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,NaN,NaN,2008-04-18,1,3,219,116
128,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.5,0,5,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,NaN,NaN,2008-04-18,1,4,218,115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248533,1473511,Narendra Modi Stadium,Royal Challengers Bengaluru,190,2,19.2,19,2,Punjab Kings,Royal Challengers Bengaluru,...,NaN,NaN,NaN,NaN,NaN,2025-06-03,0,162,28,4
248534,1473511,Narendra Modi Stadium,Royal Challengers Bengaluru,190,2,19.3,19,3,Punjab Kings,Royal Challengers Bengaluru,...,NaN,NaN,NaN,NaN,NaN,2025-06-03,6,168,22,3
248535,1473511,Narendra Modi Stadium,Royal Challengers Bengaluru,190,2,19.4,19,4,Punjab Kings,Royal Challengers Bengaluru,...,NaN,NaN,NaN,NaN,NaN,2025-06-03,4,172,18,2
248536,1473511,Narendra Modi Stadium,Royal Challengers Bengaluru,190,2,19.5,19,5,Punjab Kings,Royal Challengers Bengaluru,...,NaN,NaN,NaN,NaN,NaN,2025-06-03,6,178,12,1


Need to include another columns to our data - wickets remaining

In [455]:
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].notna().astype(int)
wickets = delivery_df.groupby('matchId')['player_dismissed'].cumsum().values
delivery_df['wickets remaining'] = 10 - wickets
delivery_df.head()

,matchId,venue,winner,total_runs_x,inning,over_ball,over,ball,batting_team,bowling_team,...,LegByes,Penalty,dismissal_kind,player_dismissed,date,total_runs_y,current_score,runs_left,balls_left,wickets remaining
124,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.1,0,1,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,0,2008-04-18,1,1,221,119,10
125,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.2,0,2,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,0,2008-04-18,1,2,220,118,10
126,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.3,0,3,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,0,2008-04-18,0,2,220,117,10
127,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.4,0,4,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,0,2008-04-18,1,3,219,116,10
128,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,222,2,0.5,0,5,Royal Challengers Bangalore,Kolkata Knight Riders,...,NaN,NaN,NaN,0,2008-04-18,1,4,218,115,10


In [456]:
# crr = runs/overs
delivery_df['crr'] = (delivery_df['current_score']*6)/(120 - delivery_df['balls_left'])

In [457]:
delivery_df['rrr'] = (delivery_df['runs_left']*6)/delivery_df['balls_left']

In [458]:
delivery_df['result'] = (delivery_df['batting_team'] == delivery_df['winner']).astype(int)

In [459]:
final_df = delivery_df[['batting_team','bowling_team','venue','runs_left','balls_left','wickets remaining','total_runs_x','crr','rrr','result']]

In [460]:
final_df = final_df.sample(final_df.shape[0])

In [461]:
final_df.sample()

,batting_team,bowling_team,venue,runs_left,balls_left,wickets remaining,total_runs_x,crr,rrr,result
127769,Kolkata Knight Riders,Delhi Daredevils,Arun Jaitley Stadium,113,46,5,219,8.594595,14.73913,0


In [462]:
final_df=final_df.dropna()

In [463]:
final_df = final_df[final_df['balls_left'] != 0]    
final_df = final_df[final_df['balls_left'] != 120]   

In [464]:
X = final_df.iloc[:,:-1]
y = final_df.iloc[:,-1]
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [465]:
X_train

,batting_team,bowling_team,venue,runs_left,balls_left,wickets remaining,total_runs_x,crr,rrr
161756,Mumbai Indians,Royal Challengers Bangalore,Sheikh Zayed Stadium,65,42,7,164,7.615385,9.285714
95117,Sunrisers Hyderabad,Chennai Super Kings,MA Chidambaram Stadium,148,74,8,209,7.956522,12.000000
176430,Kolkata Knight Riders,Sunrisers Hyderabad,Dubai International Cricket Stadium,29,27,8,115,5.548387,6.444444
57661,Mumbai Indians,Delhi Daredevils,Arun Jaitley Stadium,207,118,10,207,0.000000,10.525424
96881,Rajasthan Royals,Sunrisers Hyderabad,Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket St...,35,31,7,127,6.202247,6.774194
...,...,...,...,...,...,...,...,...,...
79061,Sunrisers Hyderabad,Kolkata Knight Riders,Rajiv Gandhi International Stadium,22,28,7,130,7.043478,4.714286
153681,Kolkata Knight Riders,Delhi Capitals,Sharjah Cricket Stadium,216,108,9,228,6.000000,12.000000
183100,Royal Challengers Bangalore,Mumbai Indians,Maharashtra Cricket Association Stadium,107,79,10,151,6.439024,8.126582
98604,Kings XI Punjab,Rajasthan Royals,Narendra Modi Stadium,6,2,4,191,9.406780,18.000000


In [466]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

trf = ColumnTransformer([
    ('trf',OneHotEncoder(sparse_output=False,drop='first'),['batting_team','bowling_team','venue'])
]
,remainder='passthrough')

In [467]:
from sklearn.linear_model import LogisticRegression


In [468]:

X_train_transformed = trf.fit_transform(X_train)


X_test_transformed = trf.transform(X_test)


model = LogisticRegression(solver='liblinear')


model.fit(X_train_transformed, y_train)


y_pred = model.predict(X_test_transformed)


from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.8042659974905897

In [469]:


# runs_left = 42
# balls_left = 30
# target_score = 185


# current_score = target_score - runs_left
# balls_bowled = 120 - balls_left


# crr = (current_score * 6) / balls_bowled if balls_bowled > 0 else 0
# rrr = (runs_left * 6) / balls_left if balls_left > 0 else 0


# made_up_situation = {
#     'batting_team': ['Chennai Super Kings'], 
#     'bowling_team': ['Mumbai Indians'],
#     'venue': ['Wankhede Stadium'],
#     'runs_left': [runs_left],
#     'balls_left': [balls_left],
#     'wickets remaining': [7],
#     'total_runs_x': [target_score], 
#     'crr': [crr],          
#     'rrr': [rrr]          
# }


# input_df = pd.DataFrame(made_up_situation)
# input_transformed = trf.transform(input_df)
# probabilities = model.predict_proba(input_transformed)

# print(f"Win Probability of Batting Team: {round(probabilities[0][1] * 100, 1)}%")
# print(f"Win Probability of Bowling Team: {round(probabilities[0][0] * 100, 1)}%")

In [470]:
import pickle


pickle.dump(trf, open('trf.pkl', 'wb'))


pickle.dump(model, open('model.pkl', 'wb'))